# 🌼 1.4 užduotis: konvoliuciniai neuroniniai tinklai su savo duomenimis (tf_flowers)

Perdarytas dėstytojo notebook'as **`cnn.ipynb`** (CIFAR-10) su kitais duomenimis.

**Duomenys:** [tf_flowers](https://www.tensorflow.org/datasets/catalog/tf_flowers) – 3 670 tikrų gėlių nuotraukų, 5 klasės (*daisy, dandelion, roses, sunflowers, tulips*), licencija CC BY 2.0 (nuotraukų autoriai nurodyti archyvo `LICENSE.txt`).
Skirtingai nei CIFAR-10 (60 000 vienodų 32×32 paveikslėlių), tai nedidelis rinkinys su skirtingo dydžio nuotraukomis – artimesnis realiai situacijai, kai savų duomenų turima nedaug.

*[Edvin Gladkich, Nojus Sadaunykas]*

> Rekomenduojama: **Runtime → Change runtime type → T4 GPU** (veikia ir su CPU, tik lėčiau).

## Kaip veikia originalus notebook'as (`cnn.ipynb`)

| Dalis | Ką daro originalas | Kodėl |
|---|---|---|
| **1. CNN** | Įkelia CIFAR-10 (50 000 + 10 000 spalvotų 32×32 vaizdų, 10 klasių), normalizuoja pikselius į [0, 1], parodo 100 pavyzdžių. | Normalizavimas pagreitina ir stabilizuoja mokymą. |
| | Modelis: `Conv2D(32, 3×3)` → `MaxPooling(2×2)` → `Conv2D(64)` → `MaxPooling` → `Conv2D(64)` → `Flatten` → `Dense(64)` → `Dense(10)`. | Konvoliucijos sluoksniai mokosi vietinių požymių (kraštų, tekstūrų, formų); pooling mažina dydį ir suteikia atsparumą poslinkiams; Dense sluoksniai iš požymių nusprendžia klasę. |
| | `SparseCategoricalCrossentropy(from_logits=True)`, Adam, 20 epochų → **~70 %** tikslumas. | Klasės koduojamos sveikaisiais skaičiais, išvestis – logitai. |
| **2. Filtrai ir požymių žemėlapiai** | Pirmojo sluoksnio filtrų svoriai (3×3×3) parodomi kaip paveikslėliai; per `tf.keras.Model(inputs, layers[0].output)` gaunami vieno vaizdo požymių žemėlapiai (feature maps). | Matyti, ką tinklas „mato“: kiekvienas filtras išryškina skirtingą vaizdo savybę. |
| | Vienas požymių žemėlapis **perskaičiuojamas rankomis su NumPy**: filtras slenkamas per vaizdą, sudauginama ir sumuojama, pridedamas bias ir pritaikomas ReLU – rezultatas sutampa su Keras. | Įrodo, kad konvoliucija yra paprasta matematinė operacija. |
| **3. MLP palyginimas** | Tas pats uždavinys su paprastu tinklu `Flatten` → `Dense(32)` → `Dense(64)` → `Dense(64)` → `Dense(10)` → tik **~40 %**. | Paprastas tinklas neišnaudoja erdvinės vaizdo struktūros, todėl CNN daug pranašesnis. |

**Pastebėjimas:** originale `validation_data` yra testavimo aibė, t. y. ta pati aibė naudojama ir mokymo stebėjimui, ir galutiniam įvertinimui. Mūsų versijoje turime atskiras validavimo ir testavimo aibes.

## Ką keičiame su tf_flowers
- 5 klasės, 3 670 nuotraukų, sumažintos iki **96×96** (daugiau detalių nei 32×32).
- Duomenų mažai, todėl originalus CNN greitai **persimoko**. Pridedame antrą variantą su **duomenų augmentacija** (atsitiktiniai apvertimai, pasukimai, priartinimai) ir **Dropout**.
- Galutinis palyginimas: MLP vs CNN vs CNN su augmentacija toje pačioje testavimo aibėje, klaidų matrica ir prognozių pavyzdžiai.

# 1 DALIS – konvoliucinis neuroninis tinklas

In [ ]:
#@title 📥 STEP 1: Gėlių nuotraukų atsisiuntimas ir padalijimas 70:15:15
import pathlib
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

IMG_SIZE = 96
archive = pathlib.Path(tf.keras.utils.get_file("flower_photos.tgz", origin="https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz", extract=True))
search_root = archive if archive.is_dir() else archive.parent
data_dir = next(p.parent for p in search_root.rglob("roses") if p.is_dir())

# Visos nuotraukos sumažinamos iki IMG_SIZE × IMG_SIZE; seed=42 – ta pati tvarka visuose 1.4 ir 1.5 notebook'uose
full_ds = tf.keras.utils.image_dataset_from_directory(data_dir, image_size=(IMG_SIZE, IMG_SIZE), batch_size=None,
                                                      shuffle=True, seed=42)
class_names = full_ds.class_names
images = np.stack([x.numpy().astype("uint8") for x, _ in full_ds])
labels = np.array([int(y) for _, y in full_ds]).reshape(-1, 1)  # forma (n, 1) – kaip CIFAR-10 originale

n = len(images)
i_train, i_val = int(0.70 * n), int(0.85 * n)
train_images, val_images, test_images = images[:i_train], images[i_train:i_val], images[i_val:]
train_labels, val_labels, test_labels = labels[:i_train], labels[i_train:i_val], labels[i_val:]
print("Klasės:", class_names)
print(f"train {len(train_images)}, validation {len(val_images)}, test {len(test_images)}; vaizdo forma {images.shape[1:]}")

In [ ]:
#@title 🖼️ STEP 2: Normalizavimas ir pirmos 25 nuotraukos
train_x, val_x, test_x = train_images / 255.0, val_images / 255.0, test_images / 255.0  # kaip originale: pikseliai į [0, 1]

plt.figure(figsize=(12, 12))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([]); plt.yticks([]); plt.grid(False)
    plt.imshow(train_x[i])
    plt.xlabel(class_names[train_labels[i][0]])
plt.show()

## Modelis (ta pati architektūra kaip originale)
Skiriasi tik įvesties dydis (96×96×3 vietoj 32×32×3) ir klasių skaičius (5 vietoj 10).

In [ ]:
#@title 🧠 STEP 3: CNN – originali architektūra
from tensorflow.keras import layers, models

num_classes = len(class_names)
tf.keras.utils.set_random_seed(42)

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(num_classes),
])
model.compile(optimizer="adam", loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
model.summary()  # parametrų skaičius filtrui: 3·3·įvesties kanalai + 1 (bias)

In [ ]:
#@title 🏋️ STEP 4: Mokymas ir įvertinimas
history = model.fit(train_x, train_labels, epochs=20, validation_data=(val_x, val_labels), verbose=2)

def plot_history(history, title):
    plt.plot(history.history["accuracy"], label="train accuracy")
    plt.plot(history.history["val_accuracy"], label="validation accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.ylim([0.1, 1]); plt.title(title)
    plt.legend(loc="lower right")
    plt.show()

plot_history(history, "CNN (originali architektūra)")
cnn_loss, cnn_acc = model.evaluate(test_x, test_labels, verbose=0)
print(f"Test accuracy: {cnn_acc:.3f}")

Kreivės rodo **persimokymą**: mokymo tikslumas artėja prie 100 %, o validavimo sustoja daug žemiau. Tinklas „įsimena“ ~2 600 mokymo nuotraukų, užuot išmokęs bendrų požymių.
Tipinis sprendimas mažiems duomenų rinkiniams – **duomenų augmentacija**: kiekvieną epochą tinklas mato šiek tiek pakeistas (apverstas, pasuktas, priartintas) nuotraukas, todėl įsiminti tampa sunku. Pridedame ir `Dropout`.

In [ ]:
#@title 🔁 STEP 5: CNN su duomenų augmentacija ir Dropout
tf.keras.utils.set_random_seed(42)
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="augmentation")  # veikia tik mokymo metu; prognozuojant vaizdai nekeičiami

model_aug = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    data_augmentation,
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(num_classes),
])
model_aug.compile(optimizer="adam", loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
history_aug = model_aug.fit(train_x, train_labels, epochs=40, validation_data=(val_x, val_labels), verbose=2)
plot_history(history_aug, "CNN su augmentacija ir Dropout")
aug_loss, aug_acc = model_aug.evaluate(test_x, test_labels, verbose=0)
print(f"Test accuracy: {aug_acc:.3f}")

# 2 DALIS – filtrų ir požymių žemėlapių vizualizacija
Kaip originale: pirmojo konvoliucinio sluoksnio filtrai yra 3×3×3 svorių masyvai. Juos galima parodyti kaip mažus paveikslėlius (atskirai kiekvienam spalvų kanalui) ir pritaikyti vienai nuotraukai.

In [ ]:
#@title 🔬 STEP 6: Pirmojo sluoksnio filtrai
conv_layers = [l for l in model.layers if isinstance(l, layers.Conv2D)]
filters, biases = conv_layers[0].get_weights()
print("Filtrų forma:", filters.shape, "(aukštis, plotis, įvesties kanalai, filtrų skaičius)")

ix = 1
plt.figure(figsize=(6, 4))
for filter_i in range(2):                       # kaip originale – 2 filtrai × 3 kanalai
    for channel_j, channel in enumerate("RGB"):
        ax = plt.subplot(2, 3, ix)
        ax.set_xticks([]); ax.set_yticks([])
        plt.imshow(filters[:, :, channel_j, filter_i], cmap="gray")
        ax.set_title(f"filtras {filter_i}, {channel}", fontsize=8)
        ix += 1
plt.show()

# Papildomai: visi 32 filtrai kaip RGB paveikslėliai (normalizuoti į [0, 1])
f_min, f_max = filters.min(), filters.max()
plt.figure(figsize=(8, 4))
for i in range(32):
    ax = plt.subplot(4, 8, i + 1); ax.set_xticks([]); ax.set_yticks([])
    plt.imshow((filters[:, :, :, i] - f_min) / (f_max - f_min))
plt.suptitle("Visi 32 pirmojo sluoksnio filtrai")
plt.show()

In [ ]:
#@title 📤 STEP 7: Vienos nuotraukos požymių žemėlapiai
model_layers_0 = tf.keras.Model(inputs=model.inputs, outputs=conv_layers[0].output)  # modelis, kuris grąžina pirmojo sluoksnio išvestį
img = train_x[1]
feature_maps = model_layers_0.predict(np.expand_dims(img, axis=0), verbose=0)
print("Požymių žemėlapių forma:", feature_maps.shape)

plt.figure(figsize=(3, 3)); plt.imshow(img); plt.axis("off"); plt.title(class_names[train_labels[1][0]]); plt.show()
plt.figure(figsize=(10, 10))
for i in range(16):
    ax = plt.subplot(4, 4, i + 1); ax.set_xticks([]); ax.set_yticks([])
    plt.imshow(feature_maps[0, :, :, i], cmap="gray")
plt.show()

In [ ]:
#@title 🧮 STEP 8: Konvoliucija rankomis (NumPy) ir palyginimas su Keras
k = 2
filter = filters[:, :, :, k]
bias = biases[k]

feature_map = np.zeros((img.shape[0] - filter.shape[0] + 1, img.shape[1] - filter.shape[1] + 1))
for i in range(feature_map.shape[0]):
    for j in range(feature_map.shape[1]):
        feature_map[i, j] = np.sum(img[i:i + filter.shape[0], j:j + filter.shape[1], :] * filter) + bias
feature_map[feature_map < 0] = 0  # ReLU

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(feature_map, cmap="gray"); axes[0].set_title("NumPy"); axes[0].axis("off")
axes[1].imshow(feature_maps[0, :, :, k], cmap="gray"); axes[1].set_title("Keras"); axes[1].axis("off")
plt.show()
print("Didžiausias skirtumas tarp NumPy ir Keras:", np.abs(feature_map - feature_maps[0, :, :, k]).max())

# 3 DALIS – palyginimas su paprastu daugiasluoksniu perceptronu (MLP)
Ta pati architektūra kaip originale: vaizdas ištiesinamas į vieną vektorių (96·96·3 = 27 648 reikšmių) ir paduodamas Dense sluoksniams.

In [ ]:
#@title 🧱 STEP 9: MLP
tf.keras.utils.set_random_seed(42)
model_mlp = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Flatten(),
    layers.Dense(32, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(num_classes),
])
model_mlp.compile(optimizer="adam", loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
model_mlp.summary()
history_mlp = model_mlp.fit(train_x, train_labels, epochs=15, validation_data=(val_x, val_labels), verbose=2)
plot_history(history_mlp, "MLP")
mlp_loss, mlp_acc = model_mlp.evaluate(test_x, test_labels, verbose=0)
print(f"Test accuracy: {mlp_acc:.3f}")

## Rezultatų palyginimas

In [ ]:
#@title 📊 STEP 10: Palyginimas, klaidų matrica ir prognozių pavyzdžiai
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

results = pd.DataFrame({
    "Modelis": ["MLP", "CNN (originali architektūra)", "CNN + augmentacija + Dropout"],
    "Parametrų": [model_mlp.count_params(), model.count_params(), model_aug.count_params()],
    "Test accuracy": [mlp_acc, cnn_acc, aug_acc],
}).set_index("Modelis").round(3)
display(results)

pred = np.argmax(model_aug.predict(test_x, verbose=0), axis=1)
ConfusionMatrixDisplay.from_predictions(test_labels.ravel(), pred, display_labels=class_names, xticks_rotation=45, colorbar=False)
plt.title("CNN + augmentacija: klaidų matrica")
plt.show()

plt.figure(figsize=(13, 13))
for i in range(36):
    plt.subplot(6, 6, i + 1); plt.xticks([]); plt.yticks([]); plt.grid(False)
    plt.imshow(test_x[i])
    true, guess = class_names[test_labels[i][0]], class_names[pred[i]]
    plt.xlabel(f"{true}/{guess}", color="green" if true == guess else "red", fontsize=9)
plt.show()

## Išvados
*(užpildoma paleidus notebook'ą)*